# Walk-Forward 回测 — 权益类因子 (T-30)

目标:在 `train_size=252 / test_size=63` 滚动窗口下,对 2 个 P0 因子
(`f1=流动性`, `f2=动量`) 做样本外回测,产出每个 fold 的 IC 与
long-short spread,以及跨 fold 的均值 / IR。

PIT 纪律:`embargo_days=1` 防止训练集与测试集日期重叠。

上游数据:Silver 观测 → Feature Engine → `feature_cols` 列表。
下游消费:回测 manifest 落 `data/reports/backtests/`,供 `report build` 引用。

In [ ]:
import os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
import pandas as pd, numpy as np
from pit_market.backtest import (
    WalkForwardConfig, walk_forward, write_manifest, summarize
)

In [ ]:
# 1. 加载 Silver + Features (此处用合成数据演示,真实场景从 DuckDB 读)
rng = np.random.default_rng(42)
dates = pd.bdate_range('2022-01-03', periods=1200)
features = pd.DataFrame({
    'liquidity_z': rng.normal(0, 1, 1200),
    'momentum_20': rng.normal(0, 1, 1200),
}, index=dates)
target = 0.02 * features['liquidity_z'] + 0.01 * features['momentum_20'] + 0.005 * rng.normal(0, 1, 1200)
returns = pd.DataFrame({'fwd_return_1d': target}, index=dates)

In [ ]:
# 2. 配置 walk-forward 参数
cfg = WalkForwardConfig(
    train_size=252, test_size=63, step=63,
    feature_cols=('liquidity_z', 'momentum_20'),
    min_train=126, embargo_days=1,
)

In [ ]:
# 3. 跑回测
result = walk_forward(features, returns, cfg)
print(f'folds={len(result.folds)}  ic_mean={result.aggregate_ic():.4f}')
for fold in result.folds[:3]:
    print(fold)

In [ ]:
# 4. 写 manifest 到 data/reports/backtests/
out_dir = REPO / 'data' / 'reports' / 'backtests'
out_dir.mkdir(parents=True, exist_ok=True)
summary_path = write_manifest(result, out_dir)
print('manifest →', summary_path)

In [ ]:
# 5. 汇总统计
summary = summarize(result)
summary